# FPL 26/27 — GW2 run
This is the clean GW2 runner. It leaves the old smoke-test notebook alone.

**Before running:** the patched `.py` files should already be in `src/`.
Then use **Restart Kernel → Run All**.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

START_GW = 2
MAX_GW = 6

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import load_model
from src.fpl_api import fetch_current_players, fetch_event_live

model_file = next(
    p for p in DATA.glob("*v0.5*FIXED*.xlsx")
    if not p.name.startswith("~$")
)
model = load_model(model_file)
current_players = fetch_current_players()
team_hist_base = pd.read_csv(DATA / "team_strength_25_26.csv")

print("ROOT:", ROOT)
print("MODEL:", model_file.name)
print("Live players:", current_players.shape)

/Users/bensfolderaccount/Desktop/FPL 26-27/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/bensfolderaccount/Desktop/FPL 26-27/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


ROOT: /Users/bensfolderaccount/Desktop/FPL 26-27
MODEL: FPL_26_27_Model_v0.5_Team_Fixture_Engine_FIXED.xlsx
Live players: (616, 13)


## 1. Refresh GW2 predicted lineups
Uses FFScout + RotoWire + the latest NMA predicted-lineups article.

In [2]:
from src.lineup_sources import fetch_lineup_sources, parse_ffscout, parse_rotowire, parse_nma
from src.lineup_consensus import match_source_to_players, build_consensus
from src.start_probs import build_start_probs

raw_lineups = fetch_lineup_sources(ROOT, gameweek=START_GW)
ffs = parse_ffscout(raw_lineups["ffscout"])
rw = parse_rotowire(raw_lineups["rotowire"])
nma = parse_nma(raw_lineups["nma"])

ffs_match = match_source_to_players(ffs, current_players)
rw_match = match_source_to_players(rw, current_players)
nma_match = match_source_to_players(nma, current_players)

consensus = build_consensus(current_players, ffs_match, rw_match, nma_match)
start_probs = build_start_probs(consensus)

print("FFScout rows:", len(ffs))
print("RotoWire rows:", len(rw))
print("NMA rows:", len(nma))
print("Max team start-prob error:", start_probs.groupby("Team")["Start Prob"].sum().sub(11).abs().max())

FFScout rows: 220
RotoWire rows: 220
NMA rows: 220
Max team start-prob error: 6.306066779870889e-13


## 2. Feed GW1 actual information into the priors
GW1 is deliberately weak evidence: role/minutes matter immediately, while attacking and team-strength priors are shrunk heavily toward the preseason model.

In [3]:
from src.historical_minutes import fetch_historical_minute_data, build_minute_priors
from src.current_season import append_event_to_history, update_attack_priors_from_event, build_team_event_actuals, update_team_strengths_from_event

hist_minutes_base = fetch_historical_minute_data(DATA / "cache")
event_gw1 = fetch_event_live(1, current_players=current_players)

hist_minutes = append_event_to_history(hist_minutes_base, event_gw1, current_players, gw=1)
attack_priors_updated = update_attack_priors_from_event(
    model["Attack_Priors"], event_gw1, current_players, prior_exposure=6.0
)
team_gw1 = build_team_event_actuals(event_gw1, model["Fixtures"], gw=1)
team_hist = update_team_strengths_from_event(team_hist_base, team_gw1, prior_matches=10.0)

print("GW1 live rows:", len(event_gw1))
display(team_gw1.sort_values("Event xG", ascending=False))

GW1 live rows: 610


,Team,Opponent,Event xG,Event xGA
3,Brentford,Spurs,3.91,0.57
13,Brighton,Aston Villa,3.77,0.30
16,Liverpool,Newcastle,3.01,1.58
15,Man City,Bournemouth,2.24,0.65
18,Chelsea,Fulham,2.23,1.38
4,Crystal Palace,Everton,1.97,1.12
1,Arsenal,Coventry City,1.88,0.21
6,Man Utd,Hull City,1.82,1.08
9,Ipswich Town,Sunderland,1.79,0.67
17,Newcastle,Liverpool,1.58,3.01


## 3. Rebuild minutes / appearance / defensive priors for GW2
The defender calibration itself stays trained only on 2025/26, avoiding leakage from GW1.

In [4]:
from src.minutes import build_expected_minutes
from src.appearance import build_appearance_priors
from src.horizon_minutes import build_minutes_horizon
from src.defensive_points import build_defensive_priors, build_defensive_xpts
from src.defcon_calibration import build_defcon_oof, fit_defender_calibrator

minute_priors = build_minute_priors(current_players, hist_minutes)
minutes = build_expected_minutes(start_probs, minute_priors)
appearance_priors = build_appearance_priors(current_players, hist_minutes)
minutes_horizon = build_minutes_horizon(
    minutes, appearance_priors, start_gw=START_GW, max_gw=MAX_GW
)
appearance_horizon = minutes_horizon[[
    "Player ID", "GW", "Appearance Prob", "P60", "Expected Appearance Pts"
]].copy()
defensive_priors = build_defensive_priors(current_players, hist_minutes)

dc_oof = build_defcon_oof(hist_minutes_base, shrink_k=4)
dc_calibrator = fit_defender_calibrator(dc_oof)

print("GW2-6 minutes rows:", minutes_horizon.shape)
print("Max expected-starters error:", minutes_horizon.groupby(["GW", "Team"])["Start Prob"].sum().sub(11).abs().max())

GW2-6 minutes rows: (3080, 28)
Max expected-starters error: 1.5081269566508126e-12


## 4. External attacking priors for players missing the workbook prior
This should normally hit your existing cache, so it should not take long.

In [5]:
from src.fotmob_history import fetch_external_history, match_external_priors

leagues = ["Premier League", "La Liga", "Bundesliga", "Serie A", "Ligue 1"]
external_history = fetch_external_history(DATA / "cache", leagues)

existing_priors = attack_priors_updated.merge(
    model["Players"][["Player ID", "Code"]], on="Player ID", how="left"
)
prior_by_code = existing_priors[[
    "Code", "Current xG Share Prior", "Current xA Share Prior"
]].drop_duplicates("Code")
fallback_players = current_players.merge(prior_by_code, on="Code", how="left")
fallback_players = fallback_players[fallback_players["Current xG Share Prior"].isna()].copy()
external_priors = match_external_priors(fallback_players, external_history)
print("External-prior matches:", len(external_priors))

External-prior matches: 26


## 5. Build the actual GW2–GW6 xPts horizon

In [6]:
from src.fixture_projection import project_team_fixtures
from src.attack_projection import build_attack_horizon
from src.xpts import build_xpts

fixture_horizon, fixture_fit = project_team_fixtures(
    model["Fixtures"], team_hist, model["Market_Odds"],
    start_gw=START_GW, max_gw=MAX_GW, ridge_lambda=2.0,
)
attack_horizon = build_attack_horizon(
    minutes=minutes_horizon,
    attack_priors=attack_priors_updated,
    workbook_players=model["Players"],
    fixture_horizon=fixture_horizon,
    external_priors=external_priors,
    start_gw=START_GW,
    max_gw=MAX_GW,
)
base_horizon = build_xpts(attack_horizon, appearance_horizon)

def_parts = []
for gw in range(START_GW, MAX_GW + 1):
    gw_starts = minutes_horizon[minutes_horizon["GW"] == gw].copy()
    d = build_defensive_xpts(gw_starts, defensive_priors, defcon_calibrator=dc_calibrator)
    d["GW"] = gw
    def_parts.append(d)
defensive_horizon = pd.concat(def_parts, ignore_index=True)

xpts_horizon = base_horizon.merge(
    defensive_horizon[["Player ID", "GW", "xPts DefCon", "xPts Saves"]],
    on=["Player ID", "GW"], how="left", validate="one_to_one",
)
xpts_horizon[["xPts DefCon", "xPts Saves"]] = xpts_horizon[["xPts DefCon", "xPts Saves"]].fillna(0.0)
xpts_horizon["xPts Model"] = xpts_horizon["Base xPts"] + xpts_horizon["xPts DefCon"] + xpts_horizon["xPts Saves"]

print("xPts horizon:", xpts_horizon.shape)
print("Gameweeks:", sorted(xpts_horizon["GW"].unique().tolist()))

xPts horizon: (3080, 74)
Gameweeks: [2, 3, 4, 5, 6]


## 6. Sanity-check GW2 projections

In [7]:
CURRENT_SQUAD = [
    "Raya", "Verbruggen",
    "Gabriel", "Diomande", "Tarkowski", "Virgil", "Thiaw",
    "B.Fernandes", "Rice", "Mbeumo", "Anderson", "Ndiaye",
    "Thiago", "Emersonn", "Simms",
]

missing_names = [
    name for name in CURRENT_SQUAD
    if not current_players["Player"].astype(str).str.casefold().eq(name.casefold()).any()
]
if missing_names:
    raise ValueError(
        "These squad names did not match the live FPL API: " + str(missing_names)
        + "\nSend me this error and I will fix the spelling immediately."
    )

name_lookup = {str(n).casefold(): pid for n, pid in zip(current_players["Player"], current_players["Player ID"])}
squad_ids = [int(name_lookup[name.casefold()]) for name in CURRENT_SQUAD]

gw2_view = xpts_horizon[
    (xpts_horizon["GW"] == START_GW) & (xpts_horizon["Player ID"].isin(squad_ids))
][[
    "Player", "Team", "Effective Mins", "xG", "xA",
    "Base xPts", "xPts DefCon", "xPts Saves", "xPts Model"
]].sort_values("xPts Model", ascending=False)
display(gw2_view)

squad_cost_now = current_players.loc[current_players["Player ID"].isin(squad_ids), "Current £m"].sum()
INFERRED_BANK = round(max(0.0, 100.0 - float(squad_cost_now)), 1)
print("Current-price squad cost:", round(float(squad_cost_now), 1))
print("Inferred bank for first run:", INFERRED_BANK)

,Player,Team,Effective Mins,xG,xA,Base xPts,xPts DefCon,xPts Saves,xPts Model
3,Gabriel,Arsenal,84.328229,0.072419,0.046351,4.465849,0.464036,0.000000,4.929885
403,Virgil,Liverpool,85.040144,0.088788,0.037503,3.742934,0.581498,0.000000,4.324432
459,Anderson,Man City,83.086070,0.096496,0.138167,3.102417,1.185961,0.000000,4.288377
480,B.Fernandes,Man Utd,83.315299,0.199869,0.236224,3.935177,0.275133,0.000000,4.210309
499,Thiaw,Newcastle,84.187562,0.093219,0.031145,3.616190,0.566228,0.000000,4.182419
0,Raya,Arsenal,84.871185,0.000050,0.001504,3.909065,0.000000,0.262640,4.171706
12,Rice,Arsenal,81.879403,0.094144,0.158700,3.357646,0.678803,0.000000,4.036449
481,Mbeumo,Man Utd,78.630296,0.272168,0.129280,3.979542,0.036345,0.000000,4.015887
265,Ndiaye,Everton,82.787817,0.171821,0.139299,3.409101,0.397631,0.000000,3.806732
112,Thiago,Brentford,82.647168,0.387592,0.036910,3.568406,0.048710,0.000000,3.617116


Current-price squad cost: 99.9
Inferred bank for first run: 0.1


## 7. ROLL vs transfer
`BANK` is initially inferred from current FPL prices. If your FPL transfers page shows a different **money in the bank**, replace the number and rerun this cell.

In [8]:
from src.transfer_optimizer import recommend_transfer

BANK = INFERRED_BANK
FREE_TRANSFERS = 1

transfer_result = recommend_transfer(
    xpts_horizon=xpts_horizon,
    current_players=current_players,
    current_squad=CURRENT_SQUAD,
    bank=BANK,
    free_transfers=FREE_TRANSFERS,
    start_gw=START_GW,
    max_gw=MAX_GW,
    gw_decay=0.90,
    bench_weight=0.12,
    max_transfers_per_gw=2,
    allow_hits=False,
    time_limit=60.0,
)

print("RECOMMENDATION:", transfer_result["recommendation"])
print("\nROLL vs BEST 1FT")
display(transfer_result["comparison"])
print("\nOptimal GW2-GW6 path")
display(transfer_result["optimal"]["plan"][[
    "GW", "FT Entering", "Transfers", "Out", "In", "Bank After",
    "FT Next", "XI xPts", "Captain", "Captain xPts", "Hit Cost"
]])

RECOMMENDATION: ROLL

ROLL vs BEST 1FT


,Scenario,Decision Utility,Projected xPts Utility,First GW Transfers,First GW Out,First GW In,Net vs Roll,Raw xPts vs Roll
0,ROLL,213.183945,210.099443,0,[],[],0.000000,0.000000
1,BEST 1FT,212.057062,210.279452,1,[Tarkowski],[Guéhi],-1.126883,0.180009



Optimal GW2-GW6 path


,GW,FT Entering,Transfers,Out,In,Bank After,FT Next,XI xPts,Captain,Captain xPts,Hit Cost
0,2,1,0,[],[],0.1,2,45.079391,Gabriel,4.929885,0.0
1,3,2,2,"[Raya, Diomande]","[Suzuki, Guéhi]",0.6,1,45.767892,Suzuki,5.168952,0.0
2,4,1,0,[],[],0.6,2,44.483106,Gabriel,4.792621,0.0
3,5,2,0,[],[],0.6,3,45.384699,Guéhi,4.728431,0.0
4,6,3,0,[],[],0.6,4,45.124502,Ndiaye,4.631222,0.0


## 8. Best GW2 XI and captain from your current squad
This is taken from the forced-ROLL branch, so it answers the selection question independently of whether a transfer is recommended.

In [9]:
roll_plan = transfer_result["roll"]["plan"]
roll_gw2 = roll_plan.loc[roll_plan["GW"] == START_GW].iloc[0]

print("GW2 captain:", roll_gw2["Captain"])
print("Projected XI xPts + captain:", round(float(roll_gw2["XI xPts"] + roll_gw2["Captain xPts"]), 2))
print("\nSquad under ROLL:")
print(roll_gw2["Squad"])

GW2 captain: Gabriel
Projected XI xPts + captain: 50.01

Squad under ROLL:
['Raya', 'Gabriel', 'Rice', 'Thiago', 'Verbruggen', 'Simms', 'Tarkowski', 'Ndiaye', 'Emersonn', 'Virgil', 'Anderson', 'B.Fernandes', 'Mbeumo', 'Thiaw', 'Diomande']
